In [10]:
%pwd

'c:\\Users\\DELL\\Desktop\\ChatBot\\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws\\analysis'

In [ ]:
import os
#C:\Users\DELL\Desktop\ChatBot\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws\data\test_data
os.chdir("../")
%pwd

'c:\\Users\\DELL\\Desktop\\ChatBot\\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws'

In [1]:
import json
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union

import torch
import torch.nn.functional as F
# from pinecone import Pinecone, ServerlessSpec
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer


INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "innoscan")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

# Safer defaults for 8GB RAM laptops
QWEN_CHAT_MODEL = os.getenv("QWEN_CHAT_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
QWEN_EMBED_MODEL = os.getenv("QWEN_EMBED_MODEL", "Qwen/Qwen3-Embedding-0.6B")

SUMMARY_MAX_TOKENS = int(os.getenv("SUMMARY_MAX_TOKENS", "64"))
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "350"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "40"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "4"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_EMBED_TOKENIZER = None
_EMBED_MODEL = None
_GEN_TOKENIZER = None
_GEN_MODEL = None

c:\Users\DELL\Desktop\ChatBot\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws\chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_poc_json(path: str) -> List[Dict[str, Any]]:
    path_obj = Path(path)
    all_records: List[Dict[str, Any]] = []

    if path_obj.is_file():
        json_files = [path_obj]
    elif path_obj.is_dir():
        json_files = sorted(path_obj.glob("*.json"))
        if not json_files:
            raise FileNotFoundError(f"No .json files found in '{path}'")
    else:
        raise FileNotFoundError(f"Path '{path}' does not exist.")

    for json_file in json_files:
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                raw = json.load(f)

            if isinstance(raw, dict) and "data" in raw and isinstance(raw["data"], list):
                records = raw["data"]
            elif isinstance(raw, dict):
                records = [raw]
            elif isinstance(raw, list):
                records = raw
            else:
                raise ValueError("Unsupported JSON structure.")

            for i, record in enumerate(records):
                if not isinstance(record, dict):
                    raise ValueError(f"Record at index {i} is not an object.")

                item = dict(record)
                item["_source_file"] = json_file.name
                item["_source_path"] = str(json_file.as_posix())
                all_records.append(item)

            print(f"Loaded {json_file.name}: {len(records)} records")

        except Exception as e:
            print(f"Failed to load {json_file.name}: {e}")

    print(f"Total combined records: {len(all_records)}")
    return all_records


def extract_useful_fields(
    data: Union[Dict[str, Any], List[Dict[str, Any]]]
) -> Union[Optional[Dict[str, str]], List[Dict[str, str]]]:
    normalize = lambda v: (
        ""
        if v is None
        else ", ".join(str(x).strip() for x in v if str(x).strip())
        if isinstance(v, list)
        else "true" if v is True
        else "false" if v is False
        else json.dumps(v, ensure_ascii=False, separators=(",", ":"))
        if isinstance(v, dict)
        else str(v).strip()
    )

    def convert(item: Dict[str, Any]) -> Optional[Dict[str, str]]:
        fields = {
            "raw_id": normalize(item.get("id")),
            "project_name": normalize(item.get("title")),
            "description": normalize(item.get("description")),
            "business_problem": normalize(item.get("problem")),
            "outcomes": normalize(item.get("outcome")),
            "language": normalize(item.get("language")),
            "solution_approach": normalize(item.get("approach")),
            "technologies": normalize(item.get("stack")),
            "complexity": normalize(item.get("complexity")),
            "boilerplate_enabled": normalize(item.get("boilerplate_enabled")),
            "dev_count": normalize(item.get("dev_count")),
            "keywords": normalize(item.get("skills")),
            "timeline": normalize(item.get("timeline")),
            "owner": normalize(item.get("manager")),
            "source_file": normalize(item.get("_source_file")),
            "source_path": normalize(item.get("_source_path")),
        }

        core_fields = [
            "project_name",
            "description",
            "business_problem",
            "solution_approach",
            "technologies",
            "outcomes",
        ]
        return fields if any(fields[k] for k in core_fields) else None

    if isinstance(data, dict):
        return convert(data)

    if isinstance(data, list):
        return [fields for item in data if isinstance(item, dict) for fields in [convert(item)] if fields]

    raise TypeError("extract_useful_fields expects a dict or list of dicts.")



In [3]:
def build_summary_input(fields: Dict[str, str]) -> str:
    lines = [
        ("Project", fields["project_name"]),
        ("Description", fields["description"]),
        ("Problem", fields["business_problem"]),
        ("Approach", fields["solution_approach"]),
        ("Stack", fields["technologies"]),
        ("Outcome", fields["outcomes"]),
        ("Language", fields["language"]),
        ("Skills", fields["keywords"]),
        ("Complexity", fields["complexity"]),
        ("Team size", fields["dev_count"]),
        ("Timeline", fields["timeline"]),
        ("Manager", fields["owner"]),
        ("Boilerplate", fields["boilerplate_enabled"]),
        ("raw_id",fields["raw_id"])
    ]
    return "\n".join(f"{label}: {value}" for label, value in lines if value)



In [13]:
path_data = "data/test_data"
df = load_poc_json(path_data)
df_min = extract_useful_fields(df)
df2 = build_summary_input(df_min[0])
df2

Loaded poc1.json: 1 records
Loaded poc10.json: 1 records
Loaded poc2.json: 1 records
Loaded poc3.json: 1 records
Total combined records: 4


'Project: Intelligent Ticket Classifier\nDescription: AI‑powered system to auto‑classify IT support tickets.\nProblem: Manual ticket triaging delays resolution and increases workload.\nApproach: Use NLP model (BERT) to classify tickets into predefined categories.\nStack: Python, FastAPI, HuggingFace, Azure Functions\nOutcome: Automated ticket classification with 85-95% accuracy.\nLanguage: Python\nSkills: Data Engineer, Frontend\nComplexity: Medium\nTeam size: 1\nTimeline: 14\nManager: ram\nBoilerplate: false\nraw_id: poc_001'

In [59]:
def get_qwen_generator():
    global _GEN_TOKENIZER, _GEN_MODEL

    if "_GEN_TOKENIZER" not in globals():
        _GEN_TOKENIZER = None
    if "_GEN_MODEL" not in globals():
        _GEN_MODEL = None

    if _GEN_TOKENIZER is None or _GEN_MODEL is None:
        _GEN_TOKENIZER = AutoTokenizer.from_pretrained(
            QWEN_CHAT_MODEL,
            trust_remote_code=True,
        )

        model_kwargs = {
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
        }
        if torch.cuda.is_available():
            model_kwargs["torch_dtype"] = "auto"

        _GEN_MODEL = AutoModelForCausalLM.from_pretrained(
            QWEN_CHAT_MODEL,
            **model_kwargs,
        )
        _GEN_MODEL.to(DEVICE)
        _GEN_MODEL.eval()

    return _GEN_TOKENIZER, _GEN_MODEL


def generate_summary(summary_input: str) -> str:
    tokenizer, model = get_qwen_generator()

    messages = [
        {
            "role": "system",
            "content": (
                "Write a short factual POC summary from the provided project details. "
                "Use only the given information. Do not invent missing details. "
                "Keep it concise and retrieval-friendly."
                
            ),
        },
        {"role": "user", "content": summary_input},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=SUMMARY_MAX_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

#check if it will work on 8gb 
tokenizer, model = get_qwen_generator()
print("Model:", QWEN_CHAT_MODEL)
print("Device:", next(model.parameters()).device)

In [ ]:
df2 = build_summary_input(df_min[1])
summary_gen = generate_summary(df2[0])
summary_gen

'Poc Details:\nRaw ID: 1234567890\n\nProject Name: Secure Data Transmission\n\nObjective: To ensure secure data transmission between two parties using encryption.\n\nDetails:\n\n1. **Targeted Attack**: The attack targets an application that uses TLS (Transport Layer Security) for data encryption'

In [28]:
def chunk_text(text: str, max_chars: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    step = max(1, max_chars - overlap)

    while start < len(text):
        end = min(len(text), start + max_chars)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start += step

    return chunks

In [ ]:
df2 = build_summary_input(df_min[1])
summary_gen = generate_summary(df2[0])
summary_gen
chunk_gen = chunk_text(summary_gen)
chunk_gen

['Poc Details: Raw ID: 1234567890 Project Name: Secure Data Transmission Objective: To ensure secure data transmission between two parties using encryption. Details: 1. **Targeted Attack**: The attack targets an application that uses TLS (Transport Layer Security) for data encryption']

In [30]:
def get_embedding_model():
    global _EMBED_TOKENIZER, _EMBED_MODEL

    if "_EMBED_TOKENIZER" not in globals():
        _EMBED_TOKENIZER = None
    if "_EMBED_MODEL" not in globals():
        _EMBED_MODEL = None

    if _EMBED_TOKENIZER is None or _EMBED_MODEL is None:
        _EMBED_TOKENIZER = AutoTokenizer.from_pretrained(
            QWEN_EMBED_MODEL,
            trust_remote_code=True,
        )
        _EMBED_MODEL = AutoModel.from_pretrained(
            QWEN_EMBED_MODEL,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        _EMBED_MODEL.to(DEVICE)
        _EMBED_MODEL.eval()

    return _EMBED_TOKENIZER, _EMBED_MODEL
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

In [57]:
def embed_texts(texts: List[str]) -> List[List[float]]:
    tokenizer, model = get_embedding_model()

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    if hasattr(outputs, "embeddings"):
        vectors = outputs.embeddings
    else:
        last_hidden_state = outputs.last_hidden_state if hasattr(outputs, "last_hidden_state") else outputs[0]
        vectors = mean_pool(last_hidden_state, encoded["attention_mask"])

    vectors = F.normalize(vectors, p=2, dim=1)
    return vectors.cpu().tolist()



In [58]:
embed_text = embed_texts(chunk_gen)

In [33]:
def clean_value(value) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(v).strip() for v in value if str(v).strip())
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False, separators=(",", ":"))
    return str(value).strip()


def get_file_key(file_path: Path) -> str:
    return file_path.stem.strip().lower()

def get_doc_id(file_path: Path, data: dict) -> str:
    file_key = get_file_key(file_path)
    raw_id = clean_value(data.get("id"))
    base = f"{file_key}-{raw_id}" if raw_id else file_key
    return re.sub(r"[^a-zA-Z0-9_-]+", "-", base).strip("-").lower()

# REMOVED: clean_value() - no longer needed for ID generation
# REMOVED: get_file_key() - replaced by simplified get_doc_id()

def get_doc_id(file_path: Path) -> str:
    """Use the JSON filename (without extension) as the document ID."""
    return re.sub(r"[^a-zA-Z0-9_-]+", "-", file_path.stem).strip("-").lower()

In [52]:
[item['raw_id'] for item in df_min[0:5]]

['poc_001', 'poc_010', 'poc_002', 'poc_003']

In [34]:
file_path = Path("data/test_data/poc1.json")
get_doc_id(file_path)

'poc1'

In [23]:

from pinecone import Pinecone
from pinecone import ServerlessSpec 
from langchain_pinecone import PineconeVectorStore


def get_pinecone_client() -> Pinecone:
    api_key = os.getenv("PINECONE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing PINECONE_API_KEY")
    return Pinecone(api_key=api_key)

pc = get_pinecone_client()
index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()
print(f"Index name:      {INDEX_NAME}")
print(f"Dimension:       {stats.get('dimension', 'N/A')}")
print(f"Total vectors:   {stats.get('total_vector_count', 0)}")
print(f"Namespaces:      {list(stats.get('namespaces', {}).keys())}")


def list_index_names(pc: Pinecone) -> List[str]:
    indexes = pc.list_indexes()
    if hasattr(indexes, "names"):
        return list(indexes.names())
    names = []
    for item in indexes:
        if isinstance(item, dict):
            names.append(item.get("name"))
        else:
            names.append(getattr(item, "name", None))
    return [name for name in names if name]


def ensure_pinecone_index(pc: Pinecone, dimension: int):
    if INDEX_NAME not in list_index_names(pc):
        pc.create_index(
            name=INDEX_NAME,
            dimension=dimension,
            metric="cosine",
            spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
        while INDEX_NAME not in list_index_names(pc):
            time.sleep(2)
    return pc.Index(INDEX_NAME)

Index name:      innoscan
Dimension:       1024
Total vectors:   1
Namespaces:      ['']


In [ ]:
pc = get_pinecone_client()

# Delete existing index
#pc.delete_index(INDEX_NAME)
#print(f"Deleted index '{INDEX_NAME}'")

# Recreate with correct dimension
#vector_dim = len(embed_texts(["dimension probe"])[0])
#print(f"Embedding dimension: {vector_dim}")  # will show 1024
#index = ensure_pinecone_index(pc, vector_dim)

Deleted index 'innoscan'
Embedding dimension: 1024
Creating index 'innoscan' with dimension=1024 ...
Index 'innoscan' created.
  Dimension:       1024
  Total vectors:   0
  Namespaces:      []


In [ ]:
#list_index_names()
#get_pinecone_client()
#list_index_names(pc)
#ensure_pinecone_index(get_pinecone_client(),384)

In [55]:
df_min[1]

{'raw_id': 'poc_010',
 'project_name': 'Automated Compliance Checker',
 'description': 'Scan documents or code to check compliance with company policies.',
 'business_problem': 'Manual compliance checks are slow and error-prone.',
 'outcomes': 'Instant compliance scoring and recommendations.',
 'language': 'Python',
 'solution_approach': 'Embed policies + use vector search for compliance checking.',
 'technologies': 'Python, LangChain, Elasticsearch, Gradio',
 'complexity': 'Medium',
 'boilerplate_enabled': 'false',
 'dev_count': '1',
 'keywords': 'Frontend, Data Engineer',
 'timeline': '28',
 'owner': 'ram',
 'source_file': 'poc10.json',
 'source_path': 'data/test_data/poc10.json'}

In [60]:
df2 = build_summary_input(df_min[1])
summary_gen = generate_summary(df2[0])
summary_gen
chunk_gen = chunk_text(summary_gen)
chunk_gen
embed_text = embed_texts(chunk_gen)
#doc_id = ['poc_001', 'poc_010', 'poc_002', 'poc_003']
doc_id =    'poc_010'

In [ ]:
file_path = Path("data/test_data/poc1.json")
#doc_id =    'poc_010' # get_doc_id(file_path)




def build_pinecone_records(
    doc_id: str,
    embeddings: List[List[float]],
    fields: Dict[str, str],
) -> List[dict]:
    records = []

    for i, embedding in enumerate(embeddings):
        records.append(
            {
                "id": f"{doc_id}#chunk_{i}",
                "values": embedding,
                "metadata": {
                    "chunk_id": i,
                    "text": summary_gen,
                    "project_name": fields["project_name"],
                    "owner": fields["owner"],
                    "language": fields["language"],
                    "Poc_id": fields["raw_id"],
                },
            }
        )

    return records

In [ ]:

records = build_pinecone_records(doc_id, embed_text, df_min[0])


In [ ]:
def save_to_pinecone(index, records: List[dict], namespace: str = "") -> None:
    """Upsert records to the existing Pinecone index.
    
    Default namespace is '' (empty string) to match your existing index.
    """
    if not records:
        print("No records to upsert.")
        return

    # Validate dimension before upserting
    stats = index.describe_index_stats()
    index_dim = stats.get("dimension", 0)
    record_dim = len(records[0]["values"])

    print(f"Index name:      {INDEX_NAME}")
    print(f"Index dimension: {index_dim}")
    print(f"Record dimension:{record_dim}")
    print(f"Total vectors:   {stats.get('total_vector_count', 0)}")
    print(f"Namespace:       '{namespace}'")
    print(f"Upserting:       {len(records)} records")

    if index_dim != record_dim:
        raise ValueError(
            f"Dimension mismatch: index expects {index_dim}, "
            f"but records have {record_dim}"
        )

    index.upsert(vectors=records, namespace=namespace)

    # Confirm after upsert
    stats_after = index.describe_index_stats()
    print(f"Vectors after:   {stats_after.get('total_vector_count', 0)}")
    print("Upsert complete.")
    pc = get_pinecone_client()


In [ ]:
#doc_id = ['poc_001', 'poc_010', 'poc_002', 'poc_003']
doc_id =    'poc_003'
df  = df_min[1]
df2 = build_summary_input(df)
summary_gen = generate_summary(df2[0])
summary_gen
chunk_gen = chunk_text(summary_gen)
chunk_gen
embed_text = embed_texts(chunk_gen)

records = build_pinecone_records(doc_id, embed_text,df)
index = pc.Index(INDEX_NAME)
save_to_pinecone(index, records)


Index name:      innoscan
Index dimension: 1024
Record dimension:1024
Total vectors:   2
Namespace:       ''
Upserting:       1 records
Vectors after:   3
Upsert complete.


In [73]:
summary_gen

'Project Details:\n- **Project Name:** Secure Data Management System (SDMS)\n- **Objective:** To develop an advanced data management system that ensures secure, efficient, and reliable storage of sensitive data.\n- **Scope:** The SDMS will include features such as encryption, access control, backup, disaster recovery, and compliance'

In [74]:
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use same embedding model as your pipeline
embeddings = HuggingFaceEmbeddings(model_name=QWEN_EMBED_MODEL)

# Connect to existing index (no re-ingestion needed)
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    text_key="text",       # <-- must match metadata key
    namespace="",
)

# Use as retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("To develop an advanced data management system that ensures secure,")

for doc in docs:
    print(doc.page_content[:100])
    print(doc.metadata)
    print()

c:\Users\DELL\Desktop\ChatBot\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws\chatbot\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1812.51i

Project Details:
- **Project Name:** Secure Data Management System (SDMS)
- **Objective:** To develo
{'chunk_id': 0.0, 'language': 'Python', 'owner': 'ram', 'project_name': 'Intelligent Ticket Classifier', 'raw_id': 'poc_001'}

